# 🚚 Smart Freight: Multi-Model Regression Benchmark & Training Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

This Google Colab notebook provides an end-to-end, reproducible training and model comparison pipeline for the **Smart Freight Supply Chain** platform.

### Objectives:
1. **Data Ingestion & Integrity Audit**: Enforce zero post-fulfillment data leakage on the DataCo dataset.
2. **Multi-Model Regression Suite**: Benchmark 8 distinct regression models (Linear, Regularized Ridge/Lasso, Decision Tree, Random Forest, Tuned RF, and XGBoost).
3. **Evaluation Metrics & Visualizations**: Compute MAE, MSE, RMSE, R², and plot comparative performance graphs, actual-vs-predicted scatters, residual diagnostics, and feature importances.
4. **Automated Production Model Selection**: Empirically select the winning architecture (Lowest RMSE → Lowest MAE → Highest R²).
5. **Direct Frontend Export**: Package `production_pipeline.joblib`, `model_metadata.json`, and `regression_benchmarks.json` into a ready-to-import bundle for the Smart Freight web application.


## 1. ⚙️ Setup & Dependencies
Install required machine learning, visualization, and serialization packages.


In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn xgboost joblib

import os
import json
import time
import zipfile
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

# Plotting styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150

print("✓ All dependencies imported successfully!")
print(f"✓ XGBoost available: {XGBOOST_AVAILABLE}")


## 2. 📂 Dataset Ingestion
Upload `DataCo_Cleaned-2.csv` (or `DataCoSupplyChainDataset.csv`).
If running in Colab, the cell below provides an automated upload prompt if the dataset is not yet found in the local environment.


In [ ]:
CSV_FILENAME = "DataCo_Cleaned-2.csv"

# Check if dataset exists in current working directory
if not os.path.exists(CSV_FILENAME):
    alt_paths = [
        "/content/DataCo_Cleaned-2.csv",
        "/content/DataCoSupplyChainDataset.csv",
        "DataCoSupplyChainDataset.csv"
    ]
    for p in alt_paths:
        if os.path.exists(p):
            CSV_FILENAME = p
            break

if not os.path.exists(CSV_FILENAME):
    print(f"⚠️ '{CSV_FILENAME}' not found in runtime. Please upload your dataset:")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn.endswith('.csv'):
                CSV_FILENAME = fn
                break
    except Exception as e:
        print("Upload widget error:", e)

print(f"Loading dataset from: {CSV_FILENAME} ...")
df_raw = pd.read_csv(CSV_FILENAME, encoding='latin1' if 'latin1' in CSV_FILENAME else 'utf-8')
print(f"✓ Raw dataset loaded successfully: {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns.")


## 3. 🛡️ Data Leakage Boundary & Feature Engineering
Strict pre-shipment boundary enforcement:
- **Prohibited Post-Fulfillment Attributes (Dropped)**: `Late_delivery_risk`, `Delivery Status`, `shipping date (DateOrders)`, `Shipping delays`, `delivery_delay_class`.
- **Target**: `Days for shipping (real)` (continuous shipping duration).
- **Approved Feature Vector (21 attributes)**: 12 numeric features and 9 categorical features exactly matching the Smart Freight frontend schema.


In [ ]:
# Target variable
TARGET_COL = 'Days for shipping (real)'

# 1. Prohibited Post-Fulfillment Leakage Attributes
LEAKAGE_COLUMNS = [
    'Late_delivery_risk',
    'Delivery Status',
    'shipping date (DateOrders)',
    'Shipping delays',
    'delivery_delay_class'
]

# 2. Extract timestamp features if not already present
df = df_raw.copy()
date_col = 'order date (DateOrders)'
if date_col in df.columns:
    df['order_dt'] = pd.to_datetime(df[date_col], errors='coerce')
    if 'order_month' not in df.columns:
        df['order_month'] = df['order_dt'].dt.month
    if 'order_day' not in df.columns:
        df['order_day'] = df['order_dt'].dt.day
    if 'order_weekday' not in df.columns:
        df['order_weekday'] = df['order_dt'].dt.weekday
    if 'order_hour' not in df.columns:
        df['order_hour'] = df['order_dt'].dt.hour
    if 'is_weekend' not in df.columns:
        df['is_weekend'] = df['order_weekday'].apply(lambda w: 1 if w >= 5 else 0)

# 3. Canonical 21 Feature Vector
NUMERIC_FEATURES = [
    'Days for shipment (scheduled)',
    'Order Item Quantity',
    'Sales',
    'Order Item Discount',
    'Product Price',
    'Latitude',
    'Longitude',
    'order_month',
    'order_day',
    'order_weekday',
    'order_hour',
    'is_weekend'
]

CATEGORICAL_FEATURES = [
    'Shipping Mode',
    'Type',
    'Customer Segment',
    'Market',
    'Order Region',
    'Order Country',
    'Order State',
    'Category Name',
    'Department Name'
]

FEATURE_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Drop rows missing the target
df = df.dropna(subset=[TARGET_COL])
for c in NUMERIC_FEATURES:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
for c in CATEGORICAL_FEATURES:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().fillna("Unknown")

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET_COL].copy()

print("=" * 65)
print(f"Total Clean Records:        {len(df):,}")
print(f"Feature Count:              {len(FEATURE_COLUMNS)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)")
print(f"Target Column:              '{TARGET_COL}' (Mean: {y.mean():.2f} days, Std: {y.std():.2f} days)")
print("=" * 65)


## 4. 🔀 Train / Test Split & Preprocessing Pipeline
- Split ratio: **80% Training**, **20% Test** (random seed = 42 for strict reproducibility).
- Numeric pipeline: `SimpleImputer(strategy='median')`
- Categorical pipeline: `OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False)`
- Encapsulated inside `ColumnTransformer` to ensure end-to-end inference portability.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Training set:   {len(X_train):,} samples")
print(f"Test set:       {len(X_test):,} samples")

# Build the authoritative preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False),
            CATEGORICAL_FEATURES,
        ),
    ],
    remainder="drop",
)

# Test-fit preprocessor to verify feature dimension alignment
preprocessor.fit(X_train)
transformed_feature_count = preprocessor.transform(X_train[:5]).shape[1]
print(f"✓ Preprocessor verified! Encoded feature dimensionality: {transformed_feature_count}")


## 5. 🔬 Multi-Model Regression Benchmarking
We train and benchmark 8 regression candidates:
1. **Simple Linear Regression** (Baseline: scheduled shipping days only)
2. **Multiple Linear Regression** (All features)
3. **Ridge Regression (L2 Regularization)** (alpha=1.0)
4. **Lasso Regression (L1 Regularization)** (alpha=0.01)
5. **Decision Tree Regressor** (max_depth=15)
6. **Random Forest Regressor** (n_estimators=100)
7. **Tuned Random Forest Regressor** (n_estimators=100, max_depth=15, min_samples_split=5)
8. **XGBoost Regressor** (n_estimators=100, max_depth=6, learning_rate=0.1)


In [ ]:
# Candidate Model Definitions
candidate_models = {
    "Simple Linear Regression": LinearRegression(),
    "Multiple Linear Regression": LinearRegression(),
    "Ridge Regression (L2)": Ridge(alpha=1.0, random_state=42),
    "Lasso Regression (L1)": Lasso(alpha=0.01, random_state=42, max_iter=2000),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=15, random_state=42),
    "Random Forest Regressor": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Tuned Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, max_depth=15, min_samples_split=5, random_state=42, n_jobs=-1
    )
}

if XGBOOST_AVAILABLE:
    candidate_models["XGBoost Regressor"] = xgb.XGBRegressor(
        n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1
    )

# Execution and Benchmarking Loop
benchmark_results = []
trained_pipelines = {}
predictions_dict = {}

print("Starting Model Training & Empirical Evaluation...
" + "-" * 75)

for model_name, model_inst in candidate_models.items():
    print(f"Training: {model_name:<30}", end="")
    t0 = time.time()
    
    # Simple Linear uses only the primary scheduled days feature
    if model_name == "Simple Linear Regression":
        model_preprocessor = ColumnTransformer(
            transformers=[("num", SimpleImputer(strategy="median"), ["Days for shipment (scheduled)"])],
            remainder="drop"
        )
    else:
        model_preprocessor = preprocessor

    pipeline = Pipeline(steps=[
        ("preprocessor", model_preprocessor),
        ("regressor", model_inst)
    ])
    
    pipeline.fit(X_train, y_train)
    train_time = round(time.time() - t0, 2)
    
    y_pred = pipeline.predict(X_test)
    
    # Compute metrics
    mae = round(float(mean_absolute_error(y_test, y_pred)), 4)
    mse = round(float(mean_squared_error(y_test, y_pred)), 4)
    rmse = round(float(np.sqrt(mse)), 4)
    r2 = round(float(r2_score(y_test, y_pred)), 4)
    
    benchmark_results.append({
        "name": model_name,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "train_time_sec": train_time
    })
    
    trained_pipelines[model_name] = pipeline
    predictions_dict[model_name] = y_pred
    
    print(f"| RMSE: {rmse:<6.4f} | MAE: {mae:<6.4f} | R²: {r2:<6.4f} | Time: {train_time}s")

# Rank models by: Lowest RMSE -> Lowest MAE -> Highest R²
df_benchmark = pd.DataFrame(benchmark_results)
df_benchmark = df_benchmark.sort_values(by=["rmse", "mae", "r2"], ascending=[True, True, False]).reset_index(drop=True)
df_benchmark["rank"] = df_benchmark.index + 1

print("
" + "=" * 75)
print("FINAL BENCHMARK COMPARISON MATRIX (RANKED)")
print("=" * 75)
display_cols = ["rank", "name", "rmse", "mae", "mse", "r2", "train_time_sec"]
print(df_benchmark[display_cols].to_string(index=False))


## 6. 📊 Graphical Visualizations & Performance Diagnostics
Generate comparison charts, actual vs. predicted scatters, residual distribution error charts, and feature importances.


In [ ]:
# ── Chart 1: Model Performance Comparison (RMSE, MAE, R²) ─────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot RMSE and MAE
df_sorted = df_benchmark.sort_values(by="rmse", ascending=False)
y_pos = np.arange(len(df_sorted))
bar_width = 0.35

axes[0].barh(y_pos - bar_width/2, df_sorted['rmse'], height=bar_width, color='#38bdf8', label='RMSE (Days)')
axes[0].barh(y_pos + bar_width/2, df_sorted['mae'], height=bar_width, color='#34d399', label='MAE (Days)')
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(df_sorted['name'], fontweight='bold')
axes[0].set_xlabel('Error in Days (Lower is Better)')
axes[0].set_title('Error Comparison: RMSE & MAE Across Models', fontweight='bold')
axes[0].legend(loc='lower right')
for i, v in enumerate(df_sorted['rmse']):
    axes[0].text(v + 0.02, i - bar_width/2, f"{v:.3f}", va='center', fontsize=9)

# Plot R² Scores
axes[1].barh(y_pos, df_sorted['r2'], color='#a78bfa', height=0.5)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(df_sorted['name'], fontweight='bold')
axes[1].set_xlabel('R² Score (Higher is Better)')
axes[1].set_title('Variance Explained (R² Score)', fontweight='bold')
for i, v in enumerate(df_sorted['r2']):
    axes[1].text(v + 0.02, i, f"{v:.3f}", va='center', fontsize=9)

plt.tight_layout()
plt.show()

# ── Chart 2: Actual vs. Predicted & Residual Distribution ──────────────
best_model_name = df_benchmark.iloc[0]['name']
best_preds = predictions_dict[best_model_name]
residuals = y_test - best_preds

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Actual vs Predicted Scatter (Downsampled to 200 points for clarity)
sample_idx = np.random.RandomState(42).choice(len(y_test), size=min(250, len(y_test)), replace=False)
y_test_sub = np.array(y_test)[sample_idx]
best_preds_sub = np.array(best_preds)[sample_idx]

axes[0].scatter(y_test_sub, best_preds_sub, alpha=0.6, color='#38bdf8', edgecolors='w', s=50, label='Test Sample')
min_val = min(y_test_sub.min(), best_preds_sub.min())
max_val = max(y_test_sub.max(), best_preds_sub.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal 45° Line')
axes[0].set_xlabel('Actual Shipping Days')
axes[0].set_ylabel('Predicted Shipping Days')
axes[0].set_title(f'Actual vs. Predicted ({best_model_name})', fontweight='bold')
axes[0].legend()

# Residual Distribution
sns.histplot(residuals, kde=True, ax=axes[1], color='#6366f1', bins=30)
axes[1].axvline(0, color='red', linestyle='--', label='Zero Error')
axes[1].set_xlabel('Residual Error (Actual - Predicted Days)')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Residual Error Distribution ({best_model_name})', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Chart 3: Top 20 Feature Importances ────────────────────────────────
best_pipeline = trained_pipelines[best_model_name]
regressor = best_pipeline.named_steps["regressor"]
top_features = []

if hasattr(regressor, "feature_importances_"):
    fitted_pre = best_pipeline.named_steps["preprocessor"]
    cat_names = list(fitted_pre.named_transformers_["cat"].get_feature_names_out(CATEGORICAL_FEATURES))
    all_feature_names = NUMERIC_FEATURES + cat_names
    importances = regressor.feature_importances_
    
    feat_imp = sorted(zip(all_feature_names, [float(i) for i in importances]), key=lambda x: x[1], reverse=True)
    top_features = [{"feature": f, "importance": round(imp, 6)} for f, imp in feat_imp[:20]]
    
    df_top = pd.DataFrame(top_features).sort_values(by="importance", ascending=True)
    plt.figure(figsize=(10, 6))
    plt.barh(df_top['feature'], df_top['importance'], color='#0ea5e9')
    plt.xlabel('Relative Feature Importance')
    plt.title(f'Top 20 Predictive Features ({best_model_name})', fontweight='bold')
    plt.tight_layout()
    plt.show()


## 7. 🏆 Production Model Selection & Frontend Export
We bundle:
1. `production_pipeline.joblib`: Complete trained pipeline callable as `pipeline.predict(df)`.
2. `model_metadata.json`: Authoritative metrics read by Flask UI and dashboards.
3. `regression_benchmarks.json`: Benchmark matrix across all models.
4. `smart_freight_model_bundle.zip`: Downloadable zip file for instant deployment.


In [ ]:
# Create export directory
EXPORT_DIR = "models"
os.makedirs(EXPORT_DIR, exist_ok=True)

# Winning model metadata
best_row = df_benchmark.iloc[0]
best_model_name = best_row['name']
best_pipeline = trained_pipelines[best_model_name]

print(f"🏆 Winning Production Model: '{best_model_name}'")
print(f"   RMSE: {best_row['rmse']} days | MAE: {best_row['mae']} days | R²: {best_row['r2']}")

# 1. Save production pipeline
prod_model_path = os.path.join(EXPORT_DIR, "production_pipeline.joblib")
joblib.dump(best_pipeline, prod_model_path)
print(f"✓ Saved pipeline  → {prod_model_path}")

# 2. Build authoritative model_metadata.json
model_metadata = {
    "dataset_name": "DataCo cleaned supply-chain dataset",
    "dataset_rows_after_cleaning": len(df),
    "test_size": 0.20,
    "random_state": 42,
    "target": TARGET_COL,
    "feature_count": len(FEATURE_COLUMNS),
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "selected_final_model": best_model_name,
    "metrics": {
        "mae": best_row['mae'],
        "mse": best_row['mse'],
        "rmse": best_row['rmse'],
        "r2": best_row['r2'],
        "train_time_sec": best_row['train_time_sec']
    },
    "model_comparison": df_benchmark.to_dict(orient="records"),
    "top_feature_importances": top_features,
    "evaluated_at": time.strftime("%Y-%m-%d %H:%M:%S")
}

metadata_path = os.path.join(EXPORT_DIR, "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=2)
print(f"✓ Saved metadata  → {metadata_path}")

# 3. Build regression_benchmarks.json
benchmarks_dict = {
    "models": {row['name']: row for row in df_benchmark.to_dict(orient="records")},
    "best_model": best_model_name,
    "top_features": top_features
}
benchmarks_path = os.path.join(EXPORT_DIR, "regression_benchmarks.json")
with open(benchmarks_path, "w") as f:
    json.dump(benchmarks_dict, f, indent=2)
print(f"✓ Saved benchmarks → {benchmarks_path}")

# 4. Create downloadable zip archive
ZIP_NAME = "smart_freight_model_bundle.zip"
with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(prod_model_path, arcname="models/production_pipeline.joblib")
    zipf.write(metadata_path, arcname="models/model_metadata.json")
    zipf.write(benchmarks_path, arcname="models/regression_benchmarks.json")

print("
" + "=" * 70)
print(f"📦 Model Bundle Created: {ZIP_NAME}")
print("To import into your Smart Freight web application:")
print("1. Download 'smart_freight_model_bundle.zip'")
print("2. Unzip into your smart-freight project directory (replacing the 'models/' folder)")
print("3. Run 'python app.py' — your frontend will instantly load the new model and graphs!")
print("=" * 70)

# Trigger automatic download in Google Colab environment
try:
    from google.colab import files
    files.download(ZIP_NAME)
    print("✓ Colab file download triggered!")
except Exception:
    print(f"Manual download available: {os.path.abspath(ZIP_NAME)}")
